# CYGNUS — Cross-Survey ML Confidence Analysis

**Research question:** Can we trust the confidence score given by an ML model when it classifies variable stars from a different astronomical survey?

**Author:** rachana.s@atriauniversity.edu.in  
**Date:** 2026-09-30

> **RECONSTRUCTION NOTE:** This notebook reconstructs the analysis from scratch using
> the validated results documented in `results/`. The original analysis was performed
> interactively and no original notebook was saved to the repository.
> This notebook is provided for reproducibility and should produce results that match
> `results/ml_summary.json` and `results/phase5/phase5_metrics.csv`.

---

## Experiment design

```
TESS catalog (4,563 clean records)
    ↓ group split by tess_id (80/20)
    ├── Training set → Random Forest (200 trees, balanced)
    ├── TESS test set → evaluate internal performance
    └── Gaia (500 sources, completely separate)
           → same trained model, no retraining
           → compare predicted confidence with actual correctness
```

**Features used:** orbital period (days) + primary eclipse duration (phase fraction)  
**Target (TESS):** morph_coeff < 0.5 → label 0 (detached), ≥ 0.5 → label 1 (contact)  
**Target (Gaia):** model_type proxy — TWOGAUSSIANS/ONEGAUSSIAN → 0, else → 1

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss,
    ConfusionMatrixDisplay,
)
import joblib

ROOT = Path().resolve().parent   # assumes notebook is in notebooks/
RANDOM_STATE = 42
print(f"Project root: {ROOT}")

## 1. Load the prepared feature tables

These files were created by `experiments/01_data_preparation.py` and `experiments/02_feature_engineering.py`.
Run those scripts first if the files are missing.

In [ ]:
tess = pd.read_csv(ROOT / "data" / "processed" / "tess_features.csv")
gaia = pd.read_csv(ROOT / "data" / "processed" / "gaia_features.csv")

gaia_complete = gaia.dropna(subset=["derived_primary_ecl_duration"]).copy()

print(f"TESS (complete cases): {len(tess):,} rows")
print(f"Gaia (all):            {len(gaia):,} rows")
print(f"Gaia (complete ecl):   {len(gaia_complete):,} rows  ({len(gaia)-len(gaia_complete)} missing ecl_dur)")

tess.head(3)

## 2. Feature distributions

The two surveys have very different distributions for the same features.
This is the root cause of the confidence failure we will see later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Period
axes[0].hist(np.log10(tess["period"]), bins=40, alpha=0.7, color="#4da6ff", label="TESS")
axes[0].hist(np.log10(gaia_complete["period_days"]), bins=40, alpha=0.7, color="#b47aff", label="Gaia")
axes[0].set_xlabel("log₁₀(Period / days)")
axes[0].set_ylabel("Count")
axes[0].set_title("Period distribution")
axes[0].legend()

# Eclipse duration
axes[1].hist(tess["prim_width_pf"], bins=40, alpha=0.7, color="#4da6ff", label="TESS")
axes[1].hist(gaia_complete["derived_primary_ecl_duration"], bins=40, alpha=0.7, color="#b47aff", label="Gaia")
axes[1].set_xlabel("Primary eclipse duration (phase fraction)")
axes[1].set_ylabel("Count")
axes[1].set_title("Eclipse duration distribution")
axes[1].legend()

fig.suptitle("Feature distributions: TESS vs Gaia", fontsize=14)
fig.tight_layout()
plt.show()

print(f"Period    — TESS median: {tess['period'].median():.2f} d  |  Gaia median: {gaia_complete['period_days'].median():.2f} d")
print(f"Ecl. dur. — TESS median: {tess['prim_width_pf'].median():.3f}     |  Gaia median: {gaia_complete['derived_primary_ecl_duration'].median():.3f}")

## 3. Train/test split (group-based)

Split on unique `tess_id` so the same star never appears in both train and test.
Six tess_ids have two rows (signal_id 1 and 2); the group split keeps both together.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(tess, groups=tess["tess_id"]))
tess_train = tess.iloc[train_idx].copy()
tess_test  = tess.iloc[test_idx].copy()

print(f"TESS train: {len(tess_train):,} rows  |  TESS test: {len(tess_test):,} rows")
print(f"Train label dist: {tess_train['tess_label'].value_counts().to_dict()}")
print(f"Test label dist:  {tess_test['tess_label'].value_counts().to_dict()}")

## 4. Train the Random Forest

StandardScaler is fitted on training data only, then applied to test and Gaia data.

In [ ]:
X_train = tess_train[["period", "prim_width_pf"]].values
y_train = tess_train["tess_label"].values
X_test  = tess_test[["period", "prim_width_pf"]].values
y_test  = tess_test["tess_label"].values
X_gaia  = gaia_complete[["period_days", "derived_primary_ecl_duration"]].values
y_gaia  = gaia_complete["gaia_label"].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train only
X_test_sc  = scaler.transform(X_test)         # transform test (no refit)
X_gaia_sc  = scaler.transform(X_gaia)         # same transform for Gaia

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_sc, y_train)
print("Training complete.")
print(f"Feature importance — period: {rf.feature_importances_[0]:.3f}  |  "
      f"prim_width_pf: {rf.feature_importances_[1]:.3f}")

## 5. TESS test set performance

In [ ]:
y_test_pred  = rf.predict(X_test_sc)
y_test_proba = rf.predict_proba(X_test_sc)
test_conf    = y_test_proba.max(axis=1)

print("TESS test set results:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  F1:        {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
print(f"  Mean conf: {test_conf.mean():.4f}")
print(f"  Cal. gap:  {test_conf.mean() - accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Brier:     {brier_score_loss(y_test, y_test_proba[:, 1]):.4f}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred,
    display_labels=["Detached (0)", "Contact (1)"], ax=ax, colorbar=False)
ax.set_title("TESS test — confusion matrix")
plt.tight_layout()
plt.show()

## 6. Cross-survey evaluation on Gaia

The same trained model, applied to Gaia data. No retraining.

In [ ]:
y_gaia_pred  = rf.predict(X_gaia_sc)
y_gaia_proba = rf.predict_proba(X_gaia_sc)
gaia_conf    = y_gaia_proba.max(axis=1)

print("Gaia cross-survey results (proxy labels):")
print(f"  Accuracy:  {accuracy_score(y_gaia, y_gaia_pred):.4f}  ← drops from 92.5% to 32.5%")
print(f"  F1:        {f1_score(y_gaia, y_gaia_pred, average='weighted'):.4f}")
print(f"  Mean conf: {gaia_conf.mean():.4f}  ← model still very confident")
print(f"  Cal. gap:  {gaia_conf.mean() - accuracy_score(y_gaia, y_gaia_pred):.4f}  ← huge overconfidence")
print(f"  Brier:     {brier_score_loss(y_gaia, y_gaia_proba[:, 1]):.4f}  ← worse than guessing")

# High-confidence subset
hc = gaia_conf >= 0.90
print(f"\n  High-confidence (≥0.90): {hc.sum()} sources, accuracy = {(y_gaia[hc] == y_gaia_pred[hc]).mean():.4f}")
print("  → Even when the model claims ≥90% confidence, it is right less than half the time on Gaia.")

## 7. Reliability diagram

A calibrated model should land on the diagonal: confidence = accuracy.
- TESS (blue) stays close to the diagonal.
- Gaia (purple) falls far below — the model is severely overconfident cross-survey.

In [ ]:
def cal_curve(y_true, y_pred, conf):
    bins = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.001)]
    correct = y_true == y_pred
    xs, ys, ns = [], [], []
    for lo, hi in bins:
        mask = (conf >= lo) & (conf < hi)
        n = mask.sum()
        if n < 5:
            continue
        xs.append((lo + hi) / 2)
        ys.append(correct[mask].mean())
        ns.append(n)
    return xs, ys, ns

tx, ty, tn = cal_curve(y_test, y_test_pred, test_conf)
gx, gy, gn = cal_curve(y_gaia, y_gaia_pred, gaia_conf)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.plot(tx, ty, "o-", color="#4da6ff", lw=2, ms=8, label=f"TESS test (n={len(y_test)})")
ax.plot(gx, gy, "s-", color="#b47aff", lw=2, ms=8, label=f"Gaia (n={len(y_gaia)})")

# Annotate bin sizes
for x, y, n in zip(tx, ty, tn):
    ax.annotate(f"n={n}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8, color="#4da6ff")
for x, y, n in zip(gx, gy, gn):
    ax.annotate(f"n={n}", (x, y), textcoords="offset points", xytext=(5, -12), fontsize=8, color="#b47aff")

ax.set_xlim(0.45, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Mean confidence in bin", fontsize=12)
ax.set_ylabel("Fraction correct", fontsize=12)
ax.set_title("Reliability diagram — TESS vs Gaia cross-survey", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig(ROOT / "results" / "phase5" / "reliability_diagram_tess_vs_gaia.png", dpi=150)
plt.show()
print("Saved reliability diagram.")

## 8. Additional experiments

### Experiment 1: period-only model
Remove eclipse duration from the feature set. Does the overconfidence persist?

In [ ]:
scaler1 = StandardScaler()
X_tr1 = scaler1.fit_transform(tess_train[["period"]].values)
X_te1 = scaler1.transform(tess_test[["period"]].values)
X_ga1 = scaler1.transform(gaia_complete[["period_days"]].values)

rf1 = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                               class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf1.fit(X_tr1, y_train)

pred_te1  = rf1.predict(X_te1)
proba_te1 = rf1.predict_proba(X_te1)
conf_te1  = proba_te1.max(axis=1)

pred_ga1  = rf1.predict(X_ga1)
proba_ga1 = rf1.predict_proba(X_ga1)
conf_ga1  = proba_ga1.max(axis=1)

print("Experiment 1 — period only:")
print(f"  TESS: acc={accuracy_score(y_test, pred_te1):.4f}  conf={conf_te1.mean():.4f}  "
      f"gap={conf_te1.mean()-accuracy_score(y_test, pred_te1):.4f}")
print(f"  Gaia: acc={accuracy_score(y_gaia, pred_ga1):.4f}  conf={conf_ga1.mean():.4f}  "
      f"gap={conf_ga1.mean()-accuracy_score(y_gaia, pred_ga1):.4f}")
print("  → Removing eclipse duration makes Gaia even MORE overconfident.")

### Experiment 3: Gaia filtered to period ≥ 1 day

The Gaia sample is dominated by very short-period systems (median 0.46 days).
Filtering to period ≥ 1 day gives systems that are more like the TESS training distribution.

In [ ]:
gaia_filtered = gaia_complete[gaia_complete["period_days"] >= 1.0].copy()
print(f"Gaia filtered to period ≥ 1 day: {len(gaia_filtered)} sources")

X_gf  = scaler.transform(gaia_filtered[["period_days", "derived_primary_ecl_duration"]].values)
y_gf  = gaia_filtered["gaia_label"].values
pred_gf  = rf.predict(X_gf)
proba_gf = rf.predict_proba(X_gf)
conf_gf  = proba_gf.max(axis=1)

acc_gf   = accuracy_score(y_gf, pred_gf)
hc_gf    = conf_gf >= 0.90

print(f"  Accuracy:  {acc_gf:.4f}  (vs {accuracy_score(y_gaia, y_gaia_pred):.4f} for all Gaia)")
print(f"  Cal. gap:  {conf_gf.mean() - acc_gf:.4f}")
print(f"  High-conf (≥0.90): n={hc_gf.sum()}, acc={(y_gf[hc_gf] == pred_gf[hc_gf]).mean():.4f}")
print("  → Even restricting to longer-period systems, the overconfidence persists.")

## 9. Summary

This table matches the validated results in `results/phase5/phase5_metrics.csv`.

In [ ]:
summary = pd.read_csv(ROOT / "results" / "phase5" / "phase5_metrics.csv")
summary

## 10. Conclusions

**Research question:** Can we trust the confidence score given by an ML model when it classifies variable stars from a different astronomical survey?

**Answer: No.**

| Metric | TESS (in-survey) | Gaia (cross-survey) |
|--------|-----------------|--------------------|
| Accuracy | 92.5% | 32.5% |
| Mean confidence | 94.1% | 86.0% |
| Calibration gap | +0.016 | +0.535 |
| Brier score | 0.060 | 0.488 |
| High-conf accuracy (≥0.90) | 96.6% | **48.7%** |

**Key findings:**
1. The model works well on TESS data it was trained for (92.5% accuracy, well-calibrated).
2. On Gaia data, accuracy drops to 32.5% — barely above random for a binary problem.
3. The model's **confidence does not drop to match**: mean confidence remains 86.0%.
4. When the model claims ≥90% confidence on Gaia, it is correct only 48.7% of the time.
5. The Brier score of 0.488 for Gaia is worse than the score a constant 50% guess would give (0.25 for balanced classes).
6. This pattern persists across all three experiments (period-only, two-feature, period-filtered).

**Root cause:** The primary eclipse duration feature has a median of 0.079 in TESS vs 0.284 in Gaia — a 3.6× difference. The model learned that wide eclipses → contact (label 1), but Gaia's contact systems appear to have very different eclipse durations than TESS's. When the model sees Gaia data, it confidently predicts contact for most sources — which is wrong for 67.5% of them.

**Implications:**
- Confidence scores from a single-survey model should not be used uncritically when applied cross-survey.
- Calibration must be evaluated on the target survey before trusting confidence for decision-making.
- Feature distribution checks (comparing histograms between train and test distributions) should be a standard step before cross-survey ML deployment.